In [9]:
# use first lat, last lat, I should really have flights use the last lat, lon in the ADS-B sequence where it thinks there was a flight. 
# Also first lat, last lat has to be on American FAA airspace. 

In [14]:
from datetime import date

from data_engineering.flights.flight_type import get_flights
from flights.evaluation.american_evals import continental_usa_to_continental_usa
import polars as pl

target_date = date(2026,3,1)
df_flights = get_flights(target_date)
df_flights = df_flights.filter(pl.col("ladd") | pl.col("pia"))

all_flight_counts_by_icao = df_flights.group_by("icao").agg(pl.len().alias("all_flight_count"))
continental_flight_counts_by_icao = (
    continental_usa_to_continental_usa(df_flights)
    .group_by("icao")
    .agg(pl.len().alias("continental_flight_count"))
)
continental_only_icaos = (
    all_flight_counts_by_icao
    .join(continental_flight_counts_by_icao, on="icao", how="left")
    .filter(pl.col("all_flight_count") == pl.col("continental_flight_count"))
    .select("icao")
)

df_flights = df_flights.join(continental_only_icaos, on="icao", how="inner")

In [15]:
TELEPORT_MIN_DISTANCE_KM = 50.0
EARTH_RADIUS_KM = 6371.0088

prev_lat = pl.col("prev_last_lat").radians()
prev_lon = pl.col("prev_last_lon").radians()
curr_lat = pl.col("first_lat").radians()
curr_lon = pl.col("first_lon").radians()
haversine_a = (
    ((curr_lat - prev_lat) / 2).sin().pow(2)
    + prev_lat.cos() * curr_lat.cos() * ((curr_lon - prev_lon) / 2).sin().pow(2)
)

teleport_flights = (
    df_flights
    .sort(["icao", "takeoff_time", "landing_time"])
    .with_columns(
        pl.col("landing_time").shift(1).over("icao").alias("prev_landing_time"),
        pl.col("landing_airport_ident").shift(1).over("icao").alias("prev_landing_airport_ident"),
        pl.col("last_lat").shift(1).over("icao").alias("prev_last_lat"),
        pl.col("last_lon").shift(1).over("icao").alias("prev_last_lon"),
    )
    .filter(
        pl.all_horizontal(
            pl.col("prev_last_lat").is_not_null(),
            pl.col("prev_last_lon").is_not_null(),
            pl.col("first_lat").is_not_null(),
            pl.col("first_lon").is_not_null(),
        )
    )
    .filter(
        pl.col("prev_landing_airport_ident").is_null()
        | pl.col("takeoff_airport_ident").is_null()
        | (pl.col("prev_landing_airport_ident") != pl.col("takeoff_airport_ident"))
    )
    .with_columns(
        (2 * EARTH_RADIUS_KM * haversine_a.sqrt().clip(0, 1).arcsin()).alias("endpoint_jump_km"),
        ((pl.col("takeoff_time") - pl.col("prev_landing_time")).dt.total_seconds() / 3600).alias("time_gap_hours"),
    )
    .with_columns(
        (pl.col("endpoint_jump_km") / pl.col("time_gap_hours")).alias("required_ground_speed_kmh")
    )
    .filter(pl.col("endpoint_jump_km") >= TELEPORT_MIN_DISTANCE_KM)
    .select(
        "icao",
        "prev_landing_time",
        "takeoff_time",
        "time_gap_hours",
        "endpoint_jump_km",
        "required_ground_speed_kmh",
        "prev_landing_airport_ident",
        "takeoff_airport_ident",
        "prev_last_lat",
        "prev_last_lon",
        "first_lat",
        "first_lon",
        "pia",
        "ladd",
        "aircraft_type",
        "owner",
    )
    .sort("endpoint_jump_km", descending=True)
)

teleport_icaos = (
    teleport_flights
    .group_by("icao")
    .agg(
        pl.len().alias("teleport_count"),
        pl.max("endpoint_jump_km").round(1).alias("max_endpoint_jump_km"),
        pl.min("time_gap_hours").round(2).alias("min_time_gap_hours"),
        pl.col("takeoff_time").min().alias("first_teleport_takeoff_time"),
    )
    .sort("max_endpoint_jump_km", descending=True)
)

teleport_icaos

icao,teleport_count,max_endpoint_jump_km,min_time_gap_hours,first_teleport_takeoff_time
str,u32,f64,f64,datetime[ms]
"""a921a9""",1,1252.1,4.28,2026-03-01 23:56:26.210
"""a021e9""",1,149.9,2.91,2026-03-01 04:34:58.170
"""aa4084""",1,141.9,2.64,2026-03-01 17:35:00.380
"""ab73a0""",1,115.4,4.81,2026-03-01 19:19:07.275
"""a66561""",1,100.0,2.19,2026-03-01 22:42:47.680
…,…,…,…,…
"""a46e0f""",1,63.7,4.49,2026-03-01 22:52:59.160
"""a03387""",1,60.5,1.15,2026-03-01 23:07:03.470
"""a6b59e""",1,53.1,1.59,2026-03-01 18:41:42.355


In [16]:
df_flights.filter(pl.col("icao") == "a921a9")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,first_message_time,first_lat,first_lon,first_baro_altitude_ft,first_geom_altitude_ft,last_message_time,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,datetime[ms],f64,f64,i64,i64,datetime[ms],f64,f64,i64,i64,bool,bool,bool,bool,str,str,str,str
"""a921a9""","""""","""""",2026-03-01 18:58:35.550,"""KCNO""",2026-03-01 19:39:49.740,"""KSBP""",2026-03-01 18:58:35.060,33.973526,-117.640088,650,null,2026-03-01 19:39:49.740,35.23092,-120.629824,225,null,false,true,false,false,"""GALX""","""H&L 168 LLC""","""GULFSTREAM G200""","""A2"""
"""a921a9""","""""","""""",2026-03-01 23:56:26.210,"""KJAC""",2026-03-02 01:47:29.600,"""KMEV""",2026-03-02 00:08:30.600,43.547012,-110.774763,9500,null,2026-03-02 01:47:29.600,38.991531,-119.751932,4675,null,false,true,false,false,"""GALX""","""H&L 168 LLC""","""GULFSTREAM G200""","""A2"""


In [17]:
from datetime import datetime, timedelta

from data_engineering.adsb.read_adsb import get_icaos_in_adsb, read_adsb_lz
from flights.evaluation.visual import visualize_flight

ica= "a921a9"
start_dt = datetime(target_date.year, target_date.month, target_date.day)
visualize_flight(df_flights.filter(pl.col("icao") == ica), read_adsb_lz(start_dt, end_dt=start_dt+timedelta(days=1), icaos=[ica]).collect())
5+5

10

In [7]:
df_flights.filter(pl.col("icao") == "a1c95c")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,first_message_time,first_lat,first_lon,first_baro_altitude_ft,first_geom_altitude_ft,last_message_time,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,datetime[ms],f64,f64,i64,i64,datetime[ms],f64,f64,i64,i64,bool,bool,bool,bool,str,str,str,str
"""a1c95c""","""""","""""",2026-06-13 16:21:21.450,"""KGNV""",2026-06-13 16:59:35.730,"""KVRB""",2026-06-13 16:21:18.520,29.700531,-82.327423,1500,null,2026-06-13 17:01:39.060,27.655428,-80.424596,null,null,false,true,false,false,"""F2TH""","""TVPX AIRCRAFT SOLUTIONS INC TR…","""DASSAULT Falcon 2000""","""A2"""
"""a1c95c""","""""","""""",2026-06-13 22:35:20.290,"""KBPI""",2026-06-13 23:37:14.850,"""K20V""",2026-06-13 23:05:19.910,42.088577,-110.442061,25900,null,2026-06-13 23:37:14.850,40.076111,-106.735901,16650,null,false,true,false,false,"""F2TH""","""TVPX AIRCRAFT SOLUTIONS INC TR…","""DASSAULT Falcon 2000""","""A2"""


In [ ]:
from data_engineering.adsb.read_adsb import read_adsb
from flights.evaluation.visual import visualize_flight


visualize_flight(df_flights.filter(pl.col("icao") == "a16891"), read_adsb(target_date, icaos=["a16891"]))